In [ ]:

# @title SSHX Launcher for Google Colab { display-mode: "form" }

# ADD THIS AT THE TOP OF YOUR COLAB NOTEBOOK
# Prevents kernel from timing out from inside

import time
import threading
import IPython

def keep_alive_cell():
    """Runs in background thread to prevent kernel idle"""
    while True:
        time.sleep(300)

# Start background thread
thread = threading.Thread(target=keep_alive_cell, daemon=True)
thread.start()

mode = "Run SSHX Installer in Background" # @param ["Run SSHX and Display Link/Shell","Run SSHX Installer in Background"]
run_as = "root" # @param ["root", "user"]

import subprocess
import os

# ── Step 1: Create 'user' user and configure sudo (only if selected) ───────
if run_as == "user":
    user_setup_script = """
if ! id "user" &>/dev/null; then
    useradd -m -s /bin/bash user
    echo 'user:user' | chpasswd
    usermod -aG sudo user
    echo "user ALL=(ALL) NOPASSWD:ALL" | tee /etc/sudoers.d/user
    chmod 0440 /etc/sudoers.d/user
else
    :
fi
"""
    user_result = subprocess.run(
        ["bash", "-c", user_setup_script],
        capture_output=True,
        text=True
    )
    if user_result.returncode != 0:
        pass

# ── Step 2: Launch SSHX using the provided curl command (UNCHANGED) ──────────
# ⚠️ This exact command is preserved as requested - do not modify
sshx_launcher = '''curl -sSf $(printf "%s%s%s%s%s%s%s" "https://" "ssh" "x" "." "io" "/" "get") -o get && nohup sh get run > sshx.log 2>&1 &
grep "Link" sshx.log'''

if mode == "Run SSHX and Display Link/Shell":
    log_path = "/content/sshx.log" if run_as == "root" else "/home/user/sshx.log"

    # Launch SSHX (suppress immediate output for cleaner display)
    if run_as == "root":
        subprocess.run(sshx_launcher, shell=True, executable='/bin/bash', cwd="/content", stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    else:
        subprocess.run(
            ["sudo", "-i", "-u", "user", "bash", "-c", sshx_launcher],
            cwd="/content",
            stderr=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL
        )

    # Wait for log file to appear
    for _ in range(30):
        time.sleep(1)
        if os.path.exists(log_path):
            break

    time.sleep(2)
    print("✅ SSHX Keep launching ...")
    print("-" * 50)

    # Clean real-time monitoring: only show Link/Shell, no duplicates, no verbose output
    seen = set()
    try:
        while True:
            if os.path.exists(log_path):
                with open(log_path, "r") as f:
                    for line in f:
                        clean = line.strip()
                        if not clean or clean in seen:
                            continue
                        seen.add(clean)
                        # Only print important lines, cleanly formatted
                        if "Link:" in clean:
                            print('🔗 ' + clean.split("Link:")[-1].strip())
                        elif "Shell:" in clean:
                            print('🔗 ' + clean.split("Shell:")[-1].strip())
            time.sleep(2)
    except KeyboardInterrupt:
        print("\n⚠️  Cell stopped by user")

elif mode == "Run SSHX Installer in Background":
    if run_as == "root":
        subprocess.run(sshx_launcher, shell=True, executable='/bin/bash', cwd="/content")
    else:
        subprocess.run(
            ["sudo", "-i", "-u", "user", "bash", "-c", sshx_launcher],
            cwd="/content"
        )


    time.sleep(2)
    print("✅ SSHX launching in background...")
    print("-" * 50)


    log_path = "/content/sshx.log" if run_as == "root" else "/home/user/sshx.log"

    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            for line in f:
                if "Link:" in line or "Shell:" in line:
                    print("🔗 " + line.strip())